<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 8 - RAG and Prompt Engineering to Extract Real-World Phenotypes (EPI 264)

This notebook introduces **prompt basics** using a local LLM (via Ollama + LangChain), and then applies prompts to a subset of **real (de-identified) clinical notes** prepared in Lab 6.

## Learning goals
By the end of this notebook, you will be able to:
- Send **system + user** messages to an LLM
- Use **prompt templates** (parameterized prompts) for repeatable workflows
- Apply a structured extraction prompt to real notes from our Lab 6 cohort

> Notes: We are not building embeddings or retrieval yet. This notebook is about **prompting** and **structured extraction**.

In [2]:
# -----------------------------------------------------------
# 1. Load the Ollama Model
# -----------------------------------------------------------
# This cell loads a local Large Language Model (LLM) using LangChain's
# `ChatOllama` wrapper. Make sure the model has been pulled via Ollama CLI before use.

# To explore available models, visit:
# - Ollama Library: https://ollama.com/library
# - Hugging Face (for embeddings and additional models): https://huggingface.co/models

from langchain_ollama import ChatOllama

# Define the model name — make sure this model is already downloaded using:
#   ollama pull deepseek-v2:16b
model_name = "qwen2"  # Alternatives: "qwen2", "llama3", "gpt-oss:20b" etc.

# Initialize the model with LangChain
model = ChatOllama(model=model_name)

print(f"✅ Model '{model_name}' is loaded and ready to use.")


✅ Model 'qwen2' is loaded and ready to use.


## 2. Basic Prompt Interaction (System + User Messages)

In this section, you will run a simple clinical query to confirm your local model is working.

**Key idea:**
- The **system message** sets role + behavior (“how to answer”).
- The **user message** asks the question (“what to answer”).

<img src="./images/basic_prompt.png" alt="i2b2 Logo" width="900">


In [3]:
# -----------------------------------------------------------
# 2. Run a Simple Clinical Query with System + User Prompts
# -----------------------------------------------------------
# This tests the model by asking a simple clinical question using structured messages.

from langchain_core.messages import HumanMessage, SystemMessage
from IPython.display import display, Markdown



messages = [
    SystemMessage(content=(
        "You are a knowledgeable medical provider. "
        "Provide clear, evidence-based explanations about a medical conditions."
    )),
    HumanMessage(content="What is dementia? What are its common symptoms and treatments?")
]

# Run inference
response = model.invoke(messages)

# Display result
display(Markdown("### Model Response:"))
display(Markdown(response.content))


### Model Response:

Dementia is a term used to describe a broad range of progressive disorders that affect cognitive function, such as memory, thinking, and language skills, to the point where it impacts a person's ability to perform daily activities. It is not a single disease but a collection of symptoms that can be associated with various underlying causes. The most common type of dementia is Alzheimer's disease, which accounts for about 60-80% of dementia cases. Other types include vascular dementia, Lewy body dementia, and frontotemporal dementia.

### Common Symptoms of Dementia

1. **Memory Loss**: People with dementia often struggle with remembering recent events, forgetting where they placed items, or confusing the names of people or objects. This can progress to forgetting important personal information, such as birthdays or medical histories.

2. **Difficulty with Thinking and Reasoning**: They might have trouble with planning, problem-solving, and making decisions, including managing money and organizing tasks.

3. **Language and Communication Issues**: Symptoms can include difficulty finding the right words, trouble following conversations, or losing the ability to write and read.

4. **Problems with Judgment and Perception**: This can involve making poor decisions, not recognizing the severity of a situation, or misinterpreting sensory information, leading to issues like putting on inappropriate clothing for the weather.

5. **Disorientation**: People with dementia can struggle with understanding time, place, or day of the week. This can lead to getting lost in familiar surroundings.

6. **Mood and Behavior Changes**: Dementia can cause irritability, anxiety, depression, aggression, or apathy.

7. **Physical Changes**: Some individuals may experience changes in appetite, sleep patterns, or incontinence.

### Treatments for Dementia

1. **Medication**: Cholinesterase inhibitors (e.g., donepezil, rivastigmine, galantamine) and memantine (for moderate to severe dementia) are commonly used to help manage symptoms by improving communication between nerve cells.

2. **Non-Drug Interventions**: These include cognitive training, physical exercise, a healthy diet, and social activities that can help maintain cognitive function and improve quality of life.

3. **Support Services**: This includes counseling, occupational therapy, and support for caregivers. Cognitive rehabilitation programs can also help maintain cognitive skills and functional abilities.

4. **Care Management**: Creating a supportive living environment that accommodates changes in memory and reasoning abilities is crucial. This includes simplifying tasks, using clear labels on drawers and cabinets, and maintaining a routine.

5. **Behavioral Management**: Addressing behavioral issues with patience, understanding, and appropriate interventions can help manage agitation, depression, and other behavioral symptoms.

6. **End-of-Life Care**: As dementia progresses, it's important to ensure that the individual's medical care is focused on their comfort and quality of life, often involving palliative care.

### Conclusion

Dementia is a complex condition that requires a multi-faceted approach to treatment and care. Early diagnosis can help in planning interventions and support, improving the quality of life for both the person with dementia and their caregivers. Regular medical checks and support from healthcare professionals are essential in managing the condition effectively.

## 3. Prompt Templates (Reusable Prompts)

Hard-coding prompts works for one-off queries, but research workflows need **repeatable prompts**.

In this section, you will use `ChatPromptTemplate` to:
- Define a prompt once (with placeholders)
- Reuse it across conditions / patient contexts
- Standardize outputs across runs (important for reproducibility)

<img src="./images/prompt_template.png" alt="Prompt Template" width="900">


In [5]:
# -----------------------------------------------------------
# 3.1. Create a Reusable Prompt Template (Dynamic Querying)
# -----------------------------------------------------------
# This cell demonstrates how to build a dynamic prompt template using placeholders
# for different medical conditions and patient profiles. The prompt is populated
# with variables at runtime and sent to the LLM for inference.

from langchain_core.prompts import ChatPromptTemplate

# Define input variables
patient_type = "5-year old child"
disease = "dementia"

# Define a role-based prompt using variable placeholders
messages = [
    ("system",
     "You are a knowledgeable medical provider. Provide clear, evidence-based explanations appropriate for a {patient_type} patient."),

    ("human",
     "What is {disease}? What are its common symptoms and treatments?")
]

# Create a prompt template with dynamic inputs
prompt_template = ChatPromptTemplate.from_messages(messages)

# Fill the template with actual values
prompt = prompt_template.invoke({
    "patient_type": patient_type,
    "disease": disease
})

# Run the model with the constructed prompt
response = model.invoke(prompt)

# Display the generated answer
display(Markdown("### AI Response"))
display(Markdown(response.content))


### AI Response

Dementia is a condition that makes it very hard for people to think, remember, and communicate. It's like your brain gets confused and forgets things that it should know. It happens when parts of the brain that are important for thinking and remembering get damaged.

**Common Symptoms of Dementia:**

1. **Memory Loss:** Not just forgetting where you put your keys, but having trouble remembering important things, like where you put your glasses, your name, or knowing what day it is.
2. **Trouble with Thinking and Planning:** Having a hard time making decisions, solving problems, or organizing thoughts. For example, it might be tough to decide what to wear or remember why you're doing something.
3. **Difficulty with Language:** Forgetting words or getting confused about words. Maybe you forget how to say a word or you use the wrong word.
4. **Getting Lost:** Not being able to find your way around places you are familiar with, like your home or your school.
5. **Confusion and Disorientation:** Not knowing where you are or what's happening around you. You might think it's nighttime when it's daytime or vice versa.
6. **Problems with Behavior and Mood:** Changing in how you feel or act, like becoming sad, anxious, or angry for no reason.

**Treatments for Dementia:**

1. **Medicines:** Sometimes doctors give medicines to help with the symptoms, like helping with memory loss or controlling behavior changes.
2. **Special Care:** Getting help from caregivers or joining support groups where they can talk about how to deal with the challenges of dementia.
3. **Lifestyle Changes:** Eating healthy foods, exercising, and keeping the mind active with activities like reading, puzzles, or learning new things can help manage symptoms.
4. **Support:** Families and friends can provide emotional support, which is very important for someone with dementia and their loved ones.

Remember, each person is different, and what works for one person might not work for another. It's all about finding what helps them best.

In [8]:
# -----------------------------------------------------------
# 3.2. Prompt Template for Dementia Phenotype Extraction (Yes/No)
# -----------------------------------------------------------
# Goal:
# Demonstrate that an LLM can extract a phenotype from unstructured text.
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

messages_notes = [
    ("system",
     "You are an advanced clinical phenotyping assistant. "
     "Your task is to extract key clinical details from unstructured notes in a clear, structured format.\n"
     "Rules:\n"
     "- Use ONLY information explicitly present in the note. Do NOT infer or invent.\n"
     "- If the note is administrative/non-clinical (e.g., 'created in error', 'dictated', 'reconcile meds'), still output sections 1–4, but keep them minimal.\n"
     "- Do NOT include meta statements about redaction, privacy, dates shifted, or removed identifiers.\n"),

    ("human",
     "Analyze the following clinical note:\n\n"
     "{patient_note}\n\n"
     "Extract the following sections using numbered headings and bullet points:\n"
     "1. Patient demographics (only what is stated)\n"
     "2. Chief complaints / reason for visit\n"
     "3. Current medications (only those explicitly listed)\n"
     "4. Dementia phenotype (Yes/No)\n"
     "   - Answer 'Yes' ONLY if dementia is documented (e.g., 'dementia', 'Alzheimer', "
     "'vascular dementia', 'Lewy body dementia') OR clearly stated as a prior diagnosis.\n"
     "   - If dementia is not explicitly mentioned, answer 'No'. Do NOT infer from memory complaints alone.\n"
     "   - Under section 4, include these sub-bullets (one per line):\n"
     "     • Phenotype: Yes/No\n"
     "     • Rationale: 1–2 sentences grounded ONLY in the evidence\n"
     "     • Confidence: low/medium/high\n\n"
     "Important: Keep the output cleanly separated by sections 1–4 "
     "(do not combine evidence/rationale/confidence into one line).")
]

prompt_template_notes = ChatPromptTemplate.from_messages(messages_notes)

print(">> Prompt created and ready to use.")

>> Prompt created and ready to use.


## 4. Apply Prompts to Real Notes from the Lab 6 Cohort

Now we will load:
- the cleaned notes dataset (deduplicated + restricted to the chosen 2-year window in Lab 6)
- the structured dementia flags + physician gold label

We will:
1. Merge notes with patient-level labels
2. Inspect notes for one patient
3. Run a structured extraction prompt on a selected note

> Reminder: These notes are de-identified synthetic narratives derived from original notes, designed for teaching and downstream RAG.

In [9]:
# -----------------------------------------------------------
# 4.1. Load Cleaned Notes and Structured Evaluation Dataset
# -----------------------------------------------------------
# From Lab 6:
# - lab6_clean_notes_baseline.parquet
# - lab6_structured_dementia_eval.csv
# -----------------------------------------------------------

# Purpose:
# Load the data into a pandas DataFrame and inspect the structure before decoding.

from pathlib import Path
import pandas as pd

filepath = Path("data_files")

notes = pd.read_parquet(filepath / "lab6_clean_notes_baseline.parquet")
eval_df = pd.read_csv(filepath / "lab6_structured_dementia_eval.csv")

print("Notes shape:", notes.shape)
print("Eval dataset shape:", eval_df.shape)

display(notes.head(10))
display(eval_df.head(10))


Notes shape: (2639, 7)
Eval dataset shape: (97, 6)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,The patient continues to experience shoulder p...
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,Sylvia was seen for follow-up regarding her sy...
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,The patient is a 69-year-old woman with a medi...
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,The patient presents with a large hiatal herni...
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This progress note documents ongoing cardiolog...
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,The care manager reached out to the patient an...
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient presented for a blood pressure che...
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\n\nThe patient is a 75-year-...
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presents for a one-year follow-up ...


,patient_num,dx_dementia_flag,med_dementia_flag,baseline_dementia,gold_diagnosis,gold_dementia
0,H122355386,False,False,False,MCI vs. dementia,False
1,H116896574,False,False,False,Dementia,True
2,H112639340,False,False,False,No MCI or dementia diagnosis (normal),False
3,H117728724,False,False,False,No MCI or dementia diagnosis (normal),False
4,H116273351,False,False,False,No MCI or dementia diagnosis (normal),False
5,H113296266,False,False,False,No MCI or dementia diagnosis (normal),False
6,H112117815,False,True,True,Dementia,True
7,H115421676,False,False,False,No MCI or dementia diagnosis (normal),False
8,H115051235,False,False,False,No MCI or dementia diagnosis (normal),False
9,H111969872,False,False,False,Dementia,True


In [10]:
# -----------------------------------------------------------
# 4.2. Merge Notes with Structured & Gold Labels
# -----------------------------------------------------------

notes_eval = notes.merge(
    eval_df,
    on="patient_num",
    how="inner"
)

print("Merged dataset shape:", notes_eval.shape)
display(notes_eval.head(10))

Merged dataset shape: (2639, 12)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid,dx_dementia_flag,med_dementia_flag,baseline_dementia,gold_diagnosis,gold_dementia
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,The patient continues to experience shoulder p...,False,False,False,No MCI or dementia diagnosis (normal),False
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,Sylvia was seen for follow-up regarding her sy...,False,False,False,No MCI or dementia diagnosis (normal),False
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,The patient is a 69-year-old woman with a medi...,False,False,False,No MCI or dementia diagnosis (normal),False
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,The patient presents with a large hiatal herni...,False,False,False,No MCI or dementia diagnosis (normal),False
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This progress note documents ongoing cardiolog...,False,False,False,Dementia,True
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,The care manager reached out to the patient an...,False,False,False,MCI vs. dementia,False
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...,False,False,False,No MCI or dementia diagnosis (normal),False
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient presented for a blood pressure che...,False,False,False,No MCI or dementia diagnosis (normal),False
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\n\nThe patient is a 75-year-...,False,False,False,No MCI or dementia diagnosis (normal),False
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presents for a one-year follow-up ...,False,False,False,No MCI or dementia diagnosis (normal),False


In [11]:
# -----------------------------------------------------------
# 4.3. Inspect First 5 Notes for a Specific Patient
# -----------------------------------------------------------

from IPython.display import display, Markdown

sample_patient = "H122074591"

sample_notes = notes_eval[
    notes_eval["patient_num"] == sample_patient
].head(5)

print(f"Showing first {sample_notes.shape[0]} notes for patient {sample_patient}")

# Display metadata table
display(sample_notes[[
    "patient_num",
    "visit_id",
    "note_csn_id",
    "inpatient_note_type_dsc",
    "create_dts_shifted",
    "baseline_dementia",
    "gold_dementia"
]])

# Display text for first 10 notes
for _, row in sample_notes.iterrows():
    display(Markdown(f"""
---
### Note CSN ID: {row['note_csn_id']}
**Visit ID:** {row['visit_id']}
**Note Type:** {row['inpatient_note_type_dsc']}
**Date:** {row['create_dts_shifted']}
**Structured Dementia:** {row['baseline_dementia']}
**Gold Dementia:** {row['gold_dementia']}

---

{row['note_txt_deid']}
"""))

Showing first 5 notes for patient H122074591


,patient_num,visit_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,baseline_dementia,gold_dementia
4,H122074591,6270644869,2455733125,Progress Notes,2017-01-03 13:56:00,False,True
67,H122074591,6297519395,2507760869,Progress Notes,2017-01-24 16:25:00,False,True
303,H122074591,6325821229,2650311453,Progress Notes,2017-04-07 15:35:00,False,True
601,H122074591,6326650057,2857997493,Progress Notes,2017-07-06 15:08:00,False,True
968,H122074591,6367433259,3065515755,ED Notes,2017-09-29 21:53:00,False,True



---
### Note CSN ID: 2455733125
**Visit ID:** 6270644869
**Note Type:** Progress Notes
**Date:** 2017-01-03 13:56:00
**Structured Dementia:** False
**Gold Dementia:** True

---

This progress note documents ongoing cardiology care for an 85-year-old man with persistent atrial fibrillation since the late 1990s, hypertension, hypothyroidism, severe mitral regurgitation, and bradycardia. He is clinically stable on a rate control strategy, with anticoagulation (rivaroxaban 20 mg daily) and no history of bleeding or neurologic events. His CHADS VASC score is elevated due to age and hypertension. Creatinine clearance is 61 mL/min, supporting the current dosing.

There is a history of arrhythmic cardiac arrest attributed to proarrhythmic effects of propafenone, which led to discontinuation of the medication. Moderate to severe mitral regurgitation persists, with echocardiogram from November 2014 documenting a significantly thickened posterior mitral leaflet with flail segment and severe regurgitation. Recent imaging confirmed marked left atrial dilatation and preserved LV function (LVEF 79%). The right atrial dimension is increased, and there is mild tricuspid and pulmonary insufficiency. RV systolic pressure is estimated at 42 mmHg. Examination today remains unchanged, with a holosystolic murmur and trace pedal edema noted.

The patient remains asymptomatic from a cardiovascular perspective. He walks up to three minutes on level surfaces without symptoms, and denies chest pain, worsening dyspnea, palpitations, syncope, and orthostasis. He is mainly limited by leg fatigue, without frank claudication. Weight is stable, and appetite is good. Current vital signs show BP 118/60 mmHg and pulse 56 bpm. No evidence of acute distress. Peripheral pulses are diminished distally.

Hypothyroidism has been poorly controlled at times, with notable elevations in TSH (May 25, 2016: 25.84 uIU/mL). He takes levothyroxine (200 mcg, three tablets twice weekly due to compliance concerns). Lipid profile in May 2016 showed total cholesterol 206 mg/dL, LDL 134 mg/dL, HDL 42 mg/dL, and triglycerides 148 mg/dL.

Additional history includes asthma, squamous skin carcinoma, lactose intolerance, and a past mechanical fall two years ago resulting in right hip fracture.

The patient’s wife has mild dementia. He is a retired attorney, formerly smoked but quit over 25 years ago, and is abstinent from alcohol. Family history is significant for maternal heart failure (premature coronary artery disease) and paternal prostate cancer.

Assessment and Plan:
1. Persistent atrial fibrillation: continue rate control and rivaroxaban. Labs to be rechecked today. Advise ongoing monitoring for new symptoms potentially related to bradycardia, as pacemaker may be needed in the future.
2. Severe mitral regurgitation: Surgical intervention deferred. Continue clinical and imaging follow-up.
3. Bradycardia: No symptoms currently; continue surveillance.
4. Hypertension: Well controlled with lisinopril.
5. Hypothyroidism: Continue levothyroxine, monitor TSH more regularly.
6. General advice: Maintain walking regimen of 20 minutes daily as tolerated, avoid alcohol, monitor weight, and receive influenza vaccination. Will establish care with new primary care provider soon.

The patient is to return in six months for routine cardiology follow-up.



---
### Note CSN ID: 2507760869
**Visit ID:** 6297519395
**Note Type:** Progress Notes
**Date:** 2017-01-24 16:25:00
**Structured Dementia:** False
**Gold Dementia:** True

---

This is an update for an 85-year-old male presenting for a follow-up visit. He attended with his wife and daughter and is assisted daily with medications and activities, including support from an aide. He reports no complaints currently, but notes that his memory is poor and his right ear is blocked, though overall hearing is satisfactory. He exercises occasionally and his diet was reviewed during the visit. He takes all prescribed medications consistently and has not experienced any side effects.

Review of systems is negative for headache, chest pain, palpitations, shortness of breath, cough, gastrointestinal or urinary symptoms, rash, fever, or other acute complaints.

Vital signs and exam: Blood pressure measured at 114/62 in the right arm, sitting. Pulse is 66, respirations are 18, height is 1.803 meters, weight is 95.3 kg, BMI is 29.32. He appears alert and well, without acute distress, is oriented, and shows no inflammation or discharge from the eyes. The left ear canal and tympanic membrane are normal, but the right canal is blocked with cerumen. Neck is supple with no lymphadenopathy or thyroid masses. Lungs are clear bilaterally, heart has an irregular rhythm with a 2/6 murmur but no rubs or gallops. Extremities show no edema, clubbing, or cyanosis. Skin exam is unremarkable.

Relevant medical history and active problems addressed include persistent atrial fibrillation, hypothyroidism, Alzheimer's dementia, vitamin D deficiency, right-sided cerumen debris, and asthma.

Current medications: levothyroxine, lisinopril, rivaroxaban, sertraline. He has begun regular administration of levothyroxine and 2000 units of vitamin D daily.

Laboratory studies from a hospital outpatient visit on 1/3/2017 revealed: NT-proBNP at 944 pg/mL (within expected range for age), sodium 144 mmol/L, chloride 109 mmol/L (mild elevation), potassium 4.5 mmol/L, CO2 24 mmol/L, BUN 18 mg/dL, creatinine 1.17 mg/dL, glucose 103 mg/dL, calcium 8.8 mg/dL, eGFR 59 mL/min/1.73m2 (slightly reduced), anion gap 11 mmol/L, and magnesium 2.3 mg/dL. 

The plan is to continue current management for atrial fibrillation, hypothyroidism, dementia, asthma, vitamin D deficiency, and cerumen impaction. Orders include CBC with differential, comprehensive metabolic panel, hemoglobin A1c, lipid panel, TSH, and 25-OH vitamin D. He will return in approximately two months for repeat laboratory studies and a follow-up office visit.

His memory impairment is consistent with a diagnosis of Alzheimer's dementia, and support at home remains robust.



---
### Note CSN ID: 2650311453
**Visit ID:** 6325821229
**Note Type:** Progress Notes
**Date:** 2017-04-07 15:35:00
**Structured Dementia:** False
**Gold Dementia:** True

---

The patient is an 85-year-old male who arrived accompanied by his wife and daughter for a follow-up visit. He continues to have stable atrial fibrillation managed with present therapy. Alzheimer's disease remains an ongoing concern; with regular assistance at home, he is doing well. The patient and his daughter, who acts as his healthcare proxy and holds durable power of attorney, participated in completing the medical orders for life-sustaining treatment form, with clear expression of his preferences.

He reports experiencing sharp left chest pain lasting 15–30 minutes when falling asleep, recurring about once every three weeks. This discomfort resolves with repositioning and is not associated with shortness of breath, nausea, vomiting, or sweating. The pain was previously discussed with his cardiologist in December 2016, who considered it benign. He denies exertional chest pain.

The patient is mildly unsteady while walking and is largely uninterested in exercise despite detailed discussions about activity possibilities. He expresses limited interest in activities at his senior living residence, citing boredom and feeling isolated.

Medication review shows adherence with no reported side effects. Current prescriptions include cholecalciferol for vitamin D deficiency, levothyroxine for hypothyroidism, lisinopril, rivaroxaban, and sertraline.

Vital signs: blood pressure 116/64, pulse 75, respiratory rate 18, weight 94.9 kg, height 1.803 m, BMI 29.18, SpO2 99%. On exam, he appears alert, oriented, and well. Lungs are clear. Cardiac exam reveals irregular rhythm without murmurs, rubs, or gallops. Mild bilateral shin edema is noted. Skin exam shows actinic keratoses on arms and hands, treated with cryotherapy during the visit. Gait is slow and hesitant initially but improves with ambulation.

Laboratory findings from the outpatient visit on March 31, 2017, showed mild fasting blood sugar elevation (glucose 112 mg/dL), vitamin D deficiency (25-OH vitamin D 16 ng/mL), and normal hemoglobin A1c (5.1%). Thyroid function is stable (TSH 1.21 uIU/mL). Additional labs from January 24, 2017, confirm persistent vitamin D deficiency and mild prediabetes.

Problem-focused assessment and plan:
- Chronic atrial fibrillation: continue current management, stable.
- Hypothyroidism: continue levothyroxine therapy.
- Alzheimer's dementia: maintain current treatment, patient benefits from regular support at home. Recent insight and participation reflect fluctuating cognitive performance as described by his daughter.
- Actinic keratoses: treated with cryotherapy.
- Vitamin D deficiency: increase cholecalciferol to 3000 units daily; recheck level in three months.
- Prediabetes: mild; repeat labs in three months.

Patient denies headache, palpitations, shortness of breath, cough, nausea, vomiting, abdominal pain, bowel or urinary symptoms, rash, or fever.

Plans include continued monitoring. Patient to return for lab studies and office visit in approximately three months (around July 6, 2017).



---
### Note CSN ID: 2857997493
**Visit ID:** 6326650057
**Note Type:** Progress Notes
**Date:** 2017-07-06 15:08:00
**Structured Dementia:** False
**Gold Dementia:** True

---

This progress note documents the visit of an 86-year-old male with a diagnosis of moderate Alzheimer's dementia, along with vitamin D deficiency, hypothyroidism, and chronic atrial fibrillation. The patient arrived accompanied by his wife and daughter and reports being active, including golfing, walking, maintaining his usual diet, and reading newspapers. He is adherent to his medications with no reported side effects and denies recent headache, chest pain, palpitations, respiratory symptoms, gastrointestinal concerns, rash, or fever.

On examination, he appeared alert, well, and oriented to name, and was cooperative, though he was not always able to fully understand questions. Vital signs were stable: BP 128/62, pulse 55, respiratory rate 18, height 1.803 m, weight 94.4 kg, SpO2 98%, BMI 29.05.

Recent laboratory studies from 06/01/2017 showed persistently low vitamin D levels at 20 ng/mL (reference range 30-100 ng/mL), with other values as follows: sodium 143, chloride 103, potassium 4.6, CO2 24, BUN 19, creatinine 1.21, glucose 79, calcium 9.0, EGFR 57, anion gap 16, hemoglobin A1C 5.5, calculated mean blood glucose 111 mg/dL. EGFR was below the threshold for normal kidney function. 

Medications include cholecalciferol 4,000 units daily for vitamin D deficiency, levothyroxine for hypothyroidism, lisinopril, rivaroxaban for atrial fibrillation, and sertraline. Weight remains stable.

Plans for the next visit include checking medication administration for vitamin D and thyroid medication, and repeating laboratory studies in approximately three months (around 10/05/2017). The patient's history, allergies, medication list, and problem list were reviewed and updated.



---
### Note CSN ID: 3065515755
**Visit ID:** 6367433259
**Note Type:** ED Notes
**Date:** 2017-09-29 21:53:00
**Structured Dementia:** False
**Gold Dementia:** True

---

The patient's daughter requested a copy of the CT scan during the ED visit.


In [12]:
# -----------------------------------------------------------
# 4.4. Analyze a Real Patient Note Using note_csn_id
# -----------------------------------------------------------

from IPython.display import display, Markdown

NOTE_CSN_ID = 2650311453  # <-- set this to any note_csn_id you want

# Pull the note row
note_row = notes_eval.loc[notes_eval["note_csn_id"] == NOTE_CSN_ID].iloc[0]

# Extract the note text
patient_note_text = note_row["note_txt_deid"]

# Fill prompt
filled_prompt = prompt_template_notes.invoke({"patient_note": patient_note_text})

# Run model
clinical_response = model.invoke(filled_prompt)

# Display context + output
display(Markdown(f"""
### Note Selected
- **Patient:** {note_row['patient_num']}
- **Note CSN ID:** {note_row['note_csn_id']}
- **Note Type:** {note_row['inpatient_note_type_dsc']}
"""))

display(Markdown("### Extracted Clinical Information"))
display(Markdown(clinical_response.content))


### Note Selected
- **Patient:** H122074591
- **Note CSN ID:** 2650311453
- **Note Type:** Progress Notes


### Extracted Clinical Information

1. Patient demographics:
   - Age: 85 years
   - Gender: Male
   - Accompanied by: Wife and daughter

2. Chief complaints / reason for visit:
   - Follow-up visit for stable atrial fibrillation
   - Concerns about ongoing Alzheimer's disease
   - Chest pain experienced every three weeks
   - Limited interest in activities at senior living residence

3. Current medications:
   - Cholecalciferol (vitamin D)
   - Levothyroxine (hypothyroidism)
   - Lisinopril
   - Rivaroxaban
   - Sertraline

4. Dementia phenotype:
   - Phenotype: Yes
   - Rationale: The note explicitly states "Alzheimer's disease remains an ongoing concern"
   - Confidence: High